# Assignment 5: Baseline Action Score and Top-10 Review  (FlyRank ML, Week 4)

**Status: BUILT (scaffold). Not yet DATA-VALIDATED. Not submitted.**

**Official brief received: NO**

The official assignment brief has NOT been provided to the builder yet. Do not treat this notebook as a completed submission. Paste the official brief below, then every requirement must be implemented and checked against it.

> Rules for this notebook: no client names, domains, URLs, private queries, credentials or raw exports. Clear all outputs before committing.

## Official brief (paste here)

_PLACEHOLDER: paste the full official brief text from the FlyRank dashboard, then list each requirement and deliverable as a checklist below._

- [ ] Requirement 1
- [ ] Requirement 2

In [ ]:
from pathlib import Path
import sys

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "flyrank_ml").exists())
sys.path.insert(0, str(ROOT / "src"))

from flyrank_ml.config import BlockedError, get_hf_token, load_config, load_env_file, require_configured
load_env_file()          # reads HF_TOKEN from your local .env (never paste the token here)
cfg = load_config()
print("Project root:", ROOT)
print("HF_TOKEN set:", bool(get_hf_token()))

In [ ]:
from flyrank_ml.data.loader import load_daily_from_warehouse

try:
    daily = load_daily_from_warehouse(cfg["data"])
    print("Loaded aggregated rows:", len(daily))
except BlockedError as exc:
    daily = None
    print("BLOCKED:", exc)

## Baseline score and top-10 (aggregate columns only, pseudonymous IDs)

In [ ]:
from flyrank_ml.data.quality import clean_daily
from flyrank_ml.features import build_features
from flyrank_ml.models import baseline_score, BASELINE_DESCRIPTION
from flyrank_ml.recommendations import build_recommendations

if daily is not None:
    clean, exclusions = clean_daily(daily, "page_id", "date", ["clicks", "impressions"])
    cutoff = clean["date"].max()
    feats = build_features(clean, cutoff, window_days=cfg["experiment"]["window_days"])
    scores = baseline_score(feats)
    recs = build_recommendations(feats, scores, top_k=10)
    print(BASELINE_DESCRIPTION)
    print(recs.head(10)[["rank", "page_pseudonym", "action", "reason_codes", "score"]])
else:
    print("Skipped: no real data loaded.")

## Top-10 review
_PLACEHOLDER: manually review each of the top 10 and note whether the reason codes make sense. Do not publish URLs._